<a href="https://colab.research.google.com/github/loopBreakerr/tr-economy-ml-analysis/blob/main/Clustering_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. Veri İthalatı (Data Hand-off)
df = pd.read_csv('master_dataset_v2.csv', index_col=0, parse_dates=True)

# 2. Data Transformation (Zaman Serisinden Oransal Seriye Geçiş)
ülkeler = ['Turkey', 'Germany', 'Poland', 'Greece', 'Romania']
df_pct = pd.DataFrame()
for ulke in ülkeler:
    if ulke in df.columns:
        df_pct[ulke] = df[ulke].pct_change() * 100
df_pct.dropna(inplace=True)

# 3. KPI Extraction (Zaman Serisinden Karakter Çıkarımı)
# 60 aylık veriyi her ülke için 3 ana makroekonomik göstergeye (KPI) sıkıştırıyoruz.
kpi_data = {
    'Ortalama_Enflasyon': df_pct.mean(),
    'Volatilite_Risk': df_pct.std(),
    'Maksimum_Sok': df_pct.max()
}
df_kpi = pd.DataFrame(kpi_data)

# 4. Feature Scaling (Standardizasyon)
# K-Means mesafe (Öklid) tabanlı bir algoritmadır. Veriyi aynı ölçeğe getirmezsek
# büyük sayılar (Maksimum Şok) küçük sayıları (Ortalama) ezer. (Kritik bir Best-Practice)
scaler = StandardScaler()
kpi_scaled = scaler.fit_transform(df_kpi)

# 5. Model Execution: K-Means Clustering
# 5 ülkeyi 3 farklı ekonomik "Segment"e ayırmasını istiyoruz.
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_kpi['Cluster_ID'] = kmeans.fit_predict(kpi_scaled)

# Sonuçları konsola yazdır
print("📊 Algoritmik Kümeleme (Clustering) Sonuçları:")
print("-" * 40)
print(df_kpi[['Cluster_ID']].sort_values(by='Cluster_ID'))
print("-" * 40)

# 6. 3D Data Visualization (İnteraktif Şov)
fig = px.scatter_3d(
    df_kpi,
    x='Ortalama_Enflasyon',
    y='Volatilite_Risk',
    z='Maksimum_Sok',
    color=df_kpi['Cluster_ID'].astype(str), # Kümeleri renklendir
    text=df_kpi.index,                      # Ülke isimlerini noktaların yanına yaz
    title='Avrupa Makroekonomik Segmentasyonu (K-Means 3D)',
    labels={'color': 'Küme (Segment) ID'},
    color_discrete_sequence=px.colors.qualitative.Set1
)

# Grafiğin görsel kalibrasyonu
fig.update_traces(marker=dict(size=12, line=dict(width=2, color='DarkSlateGrey')))
fig.update_layout(scene=dict(
    xaxis_title='Ortalama Enflasyon (%)',
    yaxis_title='Volatilite (Risk) Std. Sapma',
    zaxis_title='Maksimum Kriz Şoku (%)'
))

# 3D Grafiği Ekrana Bas
fig.show()

📊 Algoritmik Kümeleme (Clustering) Sonuçları:
----------------------------------------
         Cluster_ID
Germany           0
Poland            0
Romania           0
Turkey            1
Greece            2
----------------------------------------
